# Garmin Daily Health Data Ingestion

Pull health data from the Garmin Connect API for a single date and write raw
JSON payloads to the shared `wearables_zerobus` bronze table via ZeroBus.

Each Garmin API response is stored as a VARIANT `body` alongside synthetic
HTTP headers matching the wire format used by the HealthKit iOS path.

**Prerequisites:**
- Deploy `zeroBus/dbxW_zerobus_infra` bundle first (creates schema, table, secrets).
- Push Garmin OAuth tokens to the shared secret scope:
  ```
  databricks secrets put-secret dbxw_zerobus_credentials garmin_oauth_tokens \
    --string-value "$(cat ~/.garminconnect/garmin_tokens.json)"
  ```

In [ ]:
%pip install garminconnect>=0.2.0 databricks-zerobus-ingest-sdk>=1.0.0

In [ ]:
DEFAULT_CATALOG = ""  # Must be supplied via widget or ${var.catalog} job parameter
DEFAULT_SCHEMA = "wearables"
DEFAULT_SECRET_SCOPE = "dbxw_zerobus_credentials"
DEFAULT_DEVICE_ID = "garmin_forerunner_265"

try:
    dbutils.widgets.text("catalog", DEFAULT_CATALOG, "Catalog")
    dbutils.widgets.text("schema", DEFAULT_SCHEMA, "Schema")
    dbutils.widgets.text("secret_scope", DEFAULT_SECRET_SCOPE, "Secret Scope")
    dbutils.widgets.text("target_date", "", "Target Date (YYYY-MM-DD, blank=yesterday)")
    dbutils.widgets.text("device_id", DEFAULT_DEVICE_ID, "Device ID")

    catalog = dbutils.widgets.get("catalog")
    schema = dbutils.widgets.get("schema")
    secret_scope = dbutils.widgets.get("secret_scope")
    target_date_str = dbutils.widgets.get("target_date")
    device_id = dbutils.widgets.get("device_id")
except Exception:
    catalog = DEFAULT_CATALOG
    schema = DEFAULT_SCHEMA
    secret_scope = DEFAULT_SECRET_SCOPE
    target_date_str = ""
    device_id = DEFAULT_DEVICE_ID

spark.sql(f"USE CATALOG {catalog}")

bronze_table = f"{catalog}.{schema}.wearables_zerobus"

print(f"Catalog:      {catalog}")
print(f"Schema:       {schema}")
print(f"Bronze table: {bronze_table}")
print(f"Secret scope: {secret_scope}")
print(f"Device ID:    {device_id}")

## Determine Target Date

In [ ]:
from datetime import date, timedelta

if target_date_str and target_date_str.strip():
    target_date = date.fromisoformat(target_date_str.strip())
else:
    target_date = date.today() - timedelta(days=1)

print(f"Target date: {target_date}")

## Load Garmin Tokens from Databricks Secrets

In [ ]:
from pathlib import Path

garmin_tokens_json = dbutils.secrets.get(scope=secret_scope, key="garmin_oauth_tokens")

tokenstore = Path.home() / ".garminconnect"
tokenstore.mkdir(parents=True, exist_ok=True)
token_path = tokenstore / "garmin_tokens.json"
token_path.write_text(garmin_tokens_json)

print(f"Garmin tokens written to {token_path}")

## Authenticate to Garmin Connect

In [ ]:
from garminconnect import Garmin

client = Garmin()
client.login(str(tokenstore))
print("Garmin: authenticated using saved tokens")

## Extract Daily Data from Garmin Connect API

Each API endpoint response is stored separately as raw JSON.
The bronze table receives the raw payload — parsing happens in silver.

In [ ]:
ds = target_date.isoformat()

EXTRACTORS = [
    ("daily_stats", "get_stats"),
    ("heart_rates", "get_heart_rates"),
    ("sleep", "get_sleep_data"),
    ("stress", "get_stress_data"),
    ("hrv", "get_hrv_data"),
    ("spo2", "get_spo2_data"),
    ("body_battery", "get_body_battery"),
    ("steps", "get_steps_data"),
    ("respiration", "get_respiration_data"),
]

raw_by_type = {}

for record_type, method_name in EXTRACTORS:
    try:
        raw_by_type[record_type] = getattr(client, method_name)(ds)
        print(f"  {record_type}: OK")
    except Exception as e:
        print(f"  {record_type}: FAILED ({e})")

try:
    raw_by_type["activities"] = client.get_activities_fordate(ds)
    print(f"  activities: OK")
except Exception as e:
    print(f"  activities: FAILED ({e})")

print(f"\nExtraction complete for {ds}: {len(raw_by_type)} categories")

## Build Bronze Rows (VARIANT Format)

Each Garmin API category becomes one row in `wearables_zerobus`.
The raw API response is stored in the `body` VARIANT column.
Synthetic headers mirror the wire format from the HealthKit/Connect IQ paths.

In [ ]:
import json
import uuid
from datetime import datetime, timezone


def to_bronze_row(raw_data, record_type, dev_id, target_ds):
    """Wrap a raw Garmin API response into the wearables_zerobus VARIANT schema."""
    now = datetime.now(timezone.utc).isoformat()
    body = {
        "source": "garmin_connect",
        "device_id": dev_id,
        "date": target_ds,
        "data": raw_data,
    }
    headers = {
        "Content-Type": "application/json",
        "X-Platform": "garmin_connect",
        "X-Record-Type": record_type,
        "X-Device-Id": dev_id,
        "X-Upload-Timestamp": now,
    }
    return {
        "record_id": str(uuid.uuid4()),
        "ingested_at": now,
        "body": json.dumps(body),
        "headers": json.dumps(headers),
        "record_type": record_type,
    }


bronze_rows = []
for record_type, raw_data in raw_by_type.items():
    if raw_data is None:
        continue
    bronze_rows.append(to_bronze_row(raw_data, record_type, device_id, ds))

print(f"Built {len(bronze_rows)} bronze rows")
for row in bronze_rows:
    print(f"  record_type={row['record_type']}  record_id={row['record_id'][:8]}...")

## Push Rows to ZeroBus

In [ ]:
from zerobus.sdk.shared import RecordType, StreamConfigurationOptions, TableProperties
from zerobus.sdk.sync import ZerobusSdk

zerobus_endpoint = dbutils.secrets.get(scope=secret_scope, key="zerobus_endpoint")
sp_client_id = dbutils.secrets.get(scope=secret_scope, key="client_id")
sp_client_secret = dbutils.secrets.get(scope=secret_scope, key="client_secret")
workspace_url = dbutils.secrets.get(scope=secret_scope, key="workspace_url")

target_table = dbutils.secrets.get(scope=secret_scope, key="target_table_name")
if not target_table.strip():
    target_table = bronze_table

sdk = ZerobusSdk(zerobus_endpoint, workspace_url)
table_props = TableProperties(target_table)
options = StreamConfigurationOptions(record_type=RecordType.JSON, max_inflight_requests=50)
stream = sdk.create_stream(sp_client_id, sp_client_secret, table_props, options)

offsets = []
for row in bronze_rows:
    offset = stream.ingest_record_offset(row)
    offsets.append(offset)

if offsets:
    stream.wait_for_offset(offsets[-1])

stream.close()
print(f"Pushed {len(offsets)} rows to {target_table}")

## Save Refreshed Tokens Back to Secrets

The `garminconnect` library may have refreshed the OAuth access token during the API calls above.
Write the updated token file back to Databricks Secrets so the next run uses the fresh token.

In [ ]:
refreshed_tokens = token_path.read_text()

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
w.secrets.put_secret(scope=secret_scope, key="garmin_oauth_tokens", string_value=refreshed_tokens)
print(f"Refreshed Garmin tokens saved back to secrets scope '{secret_scope}'")

## Verify

In [ ]:
count_df = spark.sql(f"""
    SELECT COUNT(*) as cnt
    FROM {bronze_table}
    WHERE headers:`X-Platform`::STRING = 'garmin_connect'
""")
display(count_df)

types_df = spark.sql(f"""
    SELECT record_type, COUNT(*) as cnt
    FROM {bronze_table}
    WHERE headers:`X-Platform`::STRING = 'garmin_connect'
    GROUP BY record_type
    ORDER BY cnt DESC
""")
display(types_df)

print(f"\nTotal Garmin records in {bronze_table}: {count_df.collect()[0]['cnt']}")